In [4]:
# ============================================================
# 0. Imports and frozen experimental configuration
# ============================================================

import json
import hashlib
import platform
import re
import sys
import unicodedata

from collections import Counter
from datetime import datetime
from pathlib import Path

from google.colab import files

DOCUMENT_ID = "D9"
DOCUMENT_NAME = (
    "Estatística do Movimento Fisiológico da População de Portugal — Ano de 1925"
)

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

SOURCE_FORMAT = ".pdf"

EXPECTED_SOURCE_SHA256 = (
    "a8fd4297949fa5b75b6e801ecd95aa9f7c15f9b08c789bc530f2266f41106cc1"
)

EXPECTED_BRANCH_B_REPRESENTATION_SHA256 = (
    "23ecda5522a5463840ef1385da3e6381ef65bb899b07382c89d35074a6dd4d48"
)

EXPECTED_PAGE_COUNT = 8

# ------------------------------------------------------------
# Frozen Stage 1 expectations.
# Used only AFTER extraction for diagnostics / Stage 4 validation.
# They are NOT disclosed to the model.
# ------------------------------------------------------------

EXPECTED_RECORD_COUNT = 19

EXPECTED_CATEGORY_COUNTS = {
    "Publication metadata": 5,
    "Index entry": 6,
    "Statistical value": 5,
    "Document structure": 3
}

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

MANDATORY_STRING_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Source Location"
]

NULLABLE_STRING_FIELDS = [
    "Unit",
    "Reporting Period"
]

ALLOWED_VALUE_TYPES = (
    str,
    int,
    float,
    type(None)
)

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

EXPECTED_TOPICS = {
    "Publication metadata": {
        "Title",
        "Reference year",
        "Publication year",
        "Publisher",
        "Institution"
    },
    "Index entry": {
        "Tabela I",
        "Tabela II",
        "Tabela III",
        "Tabela XIV",
        "Tabela LVIII",
        "Tabela LIX"
    },
    "Statistical value": {
        "Portugal area",
        "Portugal population 1911",
        "Portugal population 1920",
        "Portugal density 1920",
        "Portugal average annual population growth"
    },
    "Document structure": {
        "Rotated table",
        "Bilingual headings",
        "Historical typography"
    }
}

CRITICAL_PARENT_MARKER_PATTERNS = {
    "publication_title":
        r"movimento\s+fisiol[oó]gico",

    "reference_year":
        r"\b1925\b",

    "publication_year":
        r"\b1929\b",

    "index_heading":
        r"[íi]ndice|table\s+des\s+mati",

    "tabela_i":
        r"tabela\s+i\b",

    "tabela_ii":
        r"tabela\s+i[\s|l]*i\b",

    "portugal":
        r"\bportugal\b",

    "1911":
        r"\b1911\b",

    "1920":
        r"\b1920\b"
}

OUTPUT_DIR = Path("outputs_D9_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARENT_CHECK_PATH = (
    OUTPUT_DIR / "D9_branch_C_parent_B_equivalence_check.json"
)

NORMALISATION_CHECK_PATH = (
    OUTPUT_DIR / "D9_branch_C_normalisation_check.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR / "D9_branch_C_normalised_markdown.md"
)

REPRESENTATION_METADATA_PATH = (
    OUTPUT_DIR / "D9_branch_C_representation_metadata.json"
)

PROMPT_PATH = (
    OUTPUT_DIR / "D9_branch_C_prompt.txt"
)

EXPERIMENT_METADATA_PRE_PATH = (
    OUTPUT_DIR / "D9_branch_C_experiment_metadata_pre.json"
)

PRECHECK_PATH = (
    OUTPUT_DIR / "D9_branch_C_pre_extraction_check.json"
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR / "D9_branch_C_raw_response.txt"
)

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D9_branch_C_parsed_extraction.json"
)

STRUCTURE_CHECK_PATH = (
    OUTPUT_DIR / "D9_branch_C_structure_check.json"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR / "D9_branch_C_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR / "D9_branch_C_experiment_summary.json"
)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Parent branch:", PARENT_BRANCH)
print("Expected physical pages:", EXPECTED_PAGE_COUNT)


Document: D9
Branch: C
Parent branch: B
Expected physical pages: 8


In [5]:
# ============================================================
# 1. Upload original D9 PDF and required frozen Branch B artefacts
# ============================================================
#
# Upload exactly:
#   1) original D9 scanned PDF
#   2) D9_branch_B_structural_markdown.md
#   3) D9_branch_B_conversion_integrity.json
#
# Branch C deliberately does NOT rerun OCR.
# ============================================================

uploaded = files.upload()
names = list(uploaded.keys())

pdf_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".pdf")
]

md_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".md")
]

json_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".json")
]

if (
    len(pdf_files) != 1
    or len(md_files) != 1
    or len(json_files) != 1
):
    raise ValueError(
        "Upload exactly one PDF, one Branch B structural Markdown file, "
        "and one Branch B conversion-integrity JSON file."
    )

SOURCE_PATH = pdf_files[0]
BRANCH_B_REPRESENTATION_PATH = md_files[0]
BRANCH_B_CHECK_PATH = json_files[0]

print("Source:", SOURCE_PATH.name)
print("Branch B representation:", BRANCH_B_REPRESENTATION_PATH.name)
print("Branch B integrity:", BRANCH_B_CHECK_PATH.name)


Saving D9_branch_B_conversion_integrity.json to D9_branch_B_conversion_integrity (1).json
Saving D9_branch_B_structural_markdown.md to D9_branch_B_structural_markdown (1).md
Saving D9 - EMovimentoFisiológico1925.pdf to D9 - EMovimentoFisiológico1925 (1).pdf
Source: D9 - EMovimentoFisiológico1925 (1).pdf
Branch B representation: D9_branch_B_structural_markdown (1).md
Branch B integrity: D9_branch_B_conversion_integrity (1).json


In [6]:
# ============================================================
# 2. Verify frozen source identity and Branch B conversion provenance
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(chunk_size),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


if SOURCE_PATH.suffix.lower() != SOURCE_FORMAT:
    raise ValueError("Unexpected D9 source format.")


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

SOURCE_HASH_MATCH = (
    SOURCE_SHA256
    == EXPECTED_SOURCE_SHA256
)

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D9 PDF does not match the frozen Stage 1 source identity."
    )


with open(
    BRANCH_B_CHECK_PATH,
    "r",
    encoding="utf-8"
) as f:
    branch_b_check = json.load(f)


if branch_b_check.get("document_id") != DOCUMENT_ID:
    raise ValueError(
        "Branch B conversion-integrity artefact belongs to another document."
    )

if branch_b_check.get("branch") != "B":
    raise ValueError(
        "Uploaded conversion-integrity artefact is not from Branch B."
    )

if branch_b_check.get("source_sha256") != SOURCE_SHA256:
    raise ValueError(
        "Branch B conversion-integrity artefact refers to another D9 source."
    )

if not branch_b_check.get(
    "conversion_integrity_passed",
    False
):
    raise ValueError(
        "The frozen Branch B representation did not pass conversion integrity."
    )


SOURCE_B_MARKDOWN = (
    BRANCH_B_REPRESENTATION_PATH.read_text(
        encoding="utf-8"
    )
)

if not SOURCE_B_MARKDOWN.strip():
    raise ValueError(
        "Uploaded Branch B structural Markdown is empty."
    )


UPLOADED_BRANCH_B_SHA256 = sha256_text(
    SOURCE_B_MARKDOWN
)

BRANCH_B_HASH_MATCH = (
    UPLOADED_BRANCH_B_SHA256
    == EXPECTED_BRANCH_B_REPRESENTATION_SHA256
)

if not BRANCH_B_HASH_MATCH:
    raise ValueError(
        "Uploaded Branch B Markdown does not match the frozen final "
        "D9 Branch B representation SHA-256."
    )


print("Frozen source identity verified.")
print("Branch B conversion provenance verified.")
print("Frozen Branch B SHA-256 verified:", BRANCH_B_HASH_MATCH)


Frozen source identity verified.
Branch B conversion provenance verified.
Frozen Branch B SHA-256 verified: True


In [7]:
# ============================================================
# 3. Verify frozen Branch B parent equivalence
# ============================================================
#
# D9 Branch B includes OCR. OCR output should not be regenerated inside
# Branch C because a second OCR run could introduce a second experimental
# change. Branch C therefore starts from the exact frozen B artefact.
# ============================================================

PAGE_PATTERN = re.compile(
    r"^## Source Page (\d+)$",
    flags=re.MULTILINE
)

branch_b_page_markers = PAGE_PATTERN.findall(
    SOURCE_B_MARKDOWN
)

expected_page_markers = [
    str(page_number)
    for page_number in range(
        1,
        EXPECTED_PAGE_COUNT + 1
    )
]

BRANCH_B_PAGE_SEQUENCE_VALID = (
    branch_b_page_markers
    == expected_page_markers
)


parent_marker_status = {
    marker:
        bool(
            re.search(
                pattern,
                SOURCE_B_MARKDOWN,
                flags=re.IGNORECASE
            )
        )
    for marker, pattern
    in CRITICAL_PARENT_MARKER_PATTERNS.items()
}

BRANCH_B_MARKERS_VALID = all(
    parent_marker_status.values()
)


rotation_note_checks = {
    str(page_number):
        bool(
            re.search(
                (
                    rf"## Source Page {page_number}"
                    rf"[\s\S]*?"
                    rf"required a -?90-degree rotation for OCR reading"
                ),
                SOURCE_B_MARKDOWN,
                flags=re.IGNORECASE
            )
        )
    for page_number in [7, 8]
}

BRANCH_B_ROTATION_NOTES_VALID = all(
    rotation_note_checks.values()
)


PARENT_EQUIVALENCE_PASSED = bool(
    SOURCE_HASH_MATCH
    and branch_b_check.get(
        "conversion_integrity_passed",
        False
    )
    and BRANCH_B_HASH_MATCH
    and BRANCH_B_PAGE_SEQUENCE_VALID
    and BRANCH_B_MARKERS_VALID
    and BRANCH_B_ROTATION_NOTES_VALID
)


parent_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_sha256":
        SOURCE_SHA256,

    "source_hash_matches_frozen_identity":
        SOURCE_HASH_MATCH,

    "branch_B_conversion_integrity_passed":
        bool(
            branch_b_check.get(
                "conversion_integrity_passed",
                False
            )
        ),

    "expected_frozen_branch_B_sha256":
        EXPECTED_BRANCH_B_REPRESENTATION_SHA256,

    "uploaded_branch_B_sha256":
        UPLOADED_BRANCH_B_SHA256,

    "uploaded_branch_B_hash_matches_frozen_parent":
        BRANCH_B_HASH_MATCH,

    "branch_B_page_markers":
        branch_b_page_markers,

    "branch_B_page_sequence_valid":
        BRANCH_B_PAGE_SEQUENCE_VALID,

    "branch_B_marker_status":
        parent_marker_status,

    "branch_B_critical_markers_valid":
        BRANCH_B_MARKERS_VALID,

    "branch_B_rotation_note_checks":
        rotation_note_checks,

    "branch_B_rotation_notes_valid":
        BRANCH_B_ROTATION_NOTES_VALID,

    "parent_equivalence_method":
        (
            "Frozen Branch B artefact SHA-256 + Branch B conversion-integrity "
            "provenance; OCR is not rerun in Branch C"
        ),

    "branch_B_ocr_regeneration_attempted":
        False,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED
}


PARENT_CHECK_PATH.write_text(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    )
)


if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError(
        "D9 Branch C parent-equivalence verification failed."
    )


{
  "document_id": "D9",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "a8fd4297949fa5b75b6e801ecd95aa9f7c15f9b08c789bc530f2266f41106cc1",
  "source_hash_matches_frozen_identity": true,
  "branch_B_conversion_integrity_passed": true,
  "expected_frozen_branch_B_sha256": "23ecda5522a5463840ef1385da3e6381ef65bb899b07382c89d35074a6dd4d48",
  "uploaded_branch_B_sha256": "23ecda5522a5463840ef1385da3e6381ef65bb899b07382c89d35074a6dd4d48",
  "uploaded_branch_B_hash_matches_frozen_parent": true,
  "branch_B_page_markers": [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8"
  ],
  "branch_B_page_sequence_valid": true,
  "branch_B_marker_status": {
    "publication_title": true,
    "reference_year": true,
    "publication_year": true,
    "index_heading": true,
    "tabela_i": true,
    "tabela_ii": true,
    "portugal": true,
    "1911": true,
    "1920": true
  },
  "branch_B_critical_markers_valid": true,
  "branch_B_rotation_note_checks": {
    "7": true,


In [8]:
# ============================================================
# 4. Define conservative deterministic Branch C normalisation
# ============================================================
#
# Allowed deterministic representation changes:
# - Unicode NFKC;
# - Unicode-space standardisation;
# - typographic apostrophe standardisation;
# - dash/minus-glyph standardisation;
# - soft-hyphen removal;
# - CRLF/CR -> LF;
# - trailing whitespace removal;
# - repeated horizontal whitespace collapsed;
# - excessive blank-line runs standardised.
#
# NOT applied:
# - OCR rerun;
# - OCR spelling correction;
# - table/value reconstruction;
# - page filtering;
# - semantic rewriting;
# - unit conversion;
# - numerical calculation;
# - manual correction;
# - reference-guided repair.
# ============================================================

UNICODE_SPACE_CHARACTERS = [
    "\u00a0", "\u1680", "\u2000", "\u2001", "\u2002",
    "\u2003", "\u2004", "\u2005", "\u2006", "\u2007",
    "\u2008", "\u2009", "\u200a", "\u202f", "\u205f",
    "\u3000"
]

APOSTROPHE_REPLACEMENTS = {
    "’": "'",
    "‘": "'",
    "‛": "'",
    "´": "'",
    "`": "'"
}

DASH_REPLACEMENTS = {
    "‐": "-",
    "‑": "-",
    "‒": "-",
    "–": "-",
    "—": "-",
    "−": "-"
}


def normalise_text_representation(text):
    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(
            character,
            " "
        )

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    text = text.replace(
        "\u00ad",
        ""
    )

    text = (
        text
        .replace("\r\n", "\n")
        .replace("\r", "\n")
    )

    normalised_lines = []

    for line in text.splitlines():
        line = re.sub(
            r"[ \t\f\v]+",
            " ",
            line
        ).rstrip()

        normalised_lines.append(
            line
        )

    text = "\n".join(
        normalised_lines
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return (
        text.strip()
        + "\n"
    )


In [9]:
# ============================================================
# 5. Apply Branch C normalisation to the COMPLETE frozen B representation
# ============================================================

NORMALISED_MARKDOWN = (
    normalise_text_representation(
        SOURCE_B_MARKDOWN
    )
)

if not NORMALISED_MARKDOWN.strip():
    raise ValueError(
        "D9 Branch C normalisation produced an empty representation."
    )

print(
    "Branch B characters:",
    len(SOURCE_B_MARKDOWN)
)

print(
    "Branch C characters:",
    len(NORMALISED_MARKDOWN)
)

print(
    "Representation changed:",
    SOURCE_B_MARKDOWN != NORMALISED_MARKDOWN
)


Branch B characters: 27475
Branch C characters: 27470
Representation changed: True


In [10]:
# ============================================================
# 6. Verify Branch C normalisation integrity
# ============================================================
#
# This check is transformation-aware. The intended representation
# normalisation may change whitespace and Unicode code points, so
# raw byte equality with Branch B is not required.
# ============================================================


# ------------------------------------------------------------
# A. Page sequence
# ------------------------------------------------------------

parent_pages = PAGE_PATTERN.findall(
    SOURCE_B_MARKDOWN
)

branch_c_pages = PAGE_PATTERN.findall(
    NORMALISED_MARKDOWN
)

page_sequence_preserved = (
    parent_pages
    == branch_c_pages
    == expected_page_markers
)


# ------------------------------------------------------------
# B. Deterministic transformation reproducibility
# ------------------------------------------------------------

EXPECTED_NORMALISED_MARKDOWN = (
    normalise_text_representation(
        SOURCE_B_MARKDOWN
    )
)

deterministic_representation_verified = (
    NORMALISED_MARKDOWN
    == EXPECTED_NORMALISED_MARKDOWN
)


# ------------------------------------------------------------
# C. Critical document markers
# ------------------------------------------------------------

branch_c_marker_status = {
    marker:
        bool(
            re.search(
                pattern,
                NORMALISED_MARKDOWN,
                flags=re.IGNORECASE
            )
        )
    for marker, pattern
    in CRITICAL_PARENT_MARKER_PATTERNS.items()
}

all_critical_markers_preserved = all(
    branch_c_marker_status.values()
)


# ------------------------------------------------------------
# D. Rotation evidence inherited from B for pages 7 and 8
# ------------------------------------------------------------

branch_c_rotation_note_checks = {
    str(page_number):
        bool(
            re.search(
                (
                    rf"## Source Page {page_number}"
                    rf"[\s\S]*?"
                    rf"required a -?90-degree rotation for OCR reading"
                ),
                NORMALISED_MARKDOWN,
                flags=re.IGNORECASE
            )
        )
    for page_number in [7, 8]
}

rotation_evidence_preserved = all(
    branch_c_rotation_note_checks.values()
)


# ------------------------------------------------------------
# E. Transformation-aware quantitative token preservation
# ------------------------------------------------------------

VALUE_PATTERNS = {
    "four_digit_years":
        r"\b(?:19|20)\d{2}\b",

    "comma_grouped_numbers":
        r"(?<![\w])\d{1,3}(?:[.,]\d{3})+(?:[.,]\d+)?(?![\w])",

    "decimal_or_integer_numbers":
        r"(?<![\w])[-+]?\d+(?:[.,]\d+)?(?![\w])"
}


def canonicalise_token(token):
    token = (
        normalise_text_representation(
            token
        )
        .strip()
    )

    token = re.sub(
        r"\s+",
        "",
        token
    )

    return token.casefold()


numeric_preservation = {}

for label, pattern in VALUE_PATTERNS.items():

    before = [
        canonicalise_token(token)
        for token in re.findall(
            pattern,
            SOURCE_B_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    after = [
        canonicalise_token(token)
        for token in re.findall(
            pattern,
            NORMALISED_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    before_counter = Counter(
        before
    )

    after_counter = Counter(
        after
    )

    missing = list(
        (
            before_counter
            - after_counter
        ).elements()
    )

    added = list(
        (
            after_counter
            - before_counter
        ).elements()
    )

    numeric_preservation[label] = {
        "count_before":
            len(before),

        "count_after":
            len(after),

        "missing_token_count":
            len(missing),

        "added_token_count":
            len(added),

        "passed":
            (
                len(missing) == 0
                and len(added) == 0
            )
    }


numeric_tokens_preserved = all(
    result["passed"]
    for result
    in numeric_preservation.values()
)


# ------------------------------------------------------------
# F. No reference-guided/semantic changes
# ------------------------------------------------------------

normalisation_integrity_passed = bool(
    PARENT_EQUIVALENCE_PASSED
    and page_sequence_preserved
    and deterministic_representation_verified
    and all_critical_markers_preserved
    and rotation_evidence_preserved
    and numeric_tokens_preserved
)


normalisation_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "parent_page_markers":
        parent_pages,

    "branch_C_page_markers":
        branch_c_pages,

    "page_sequence_preserved":
        page_sequence_preserved,

    "deterministic_representation_verified":
        deterministic_representation_verified,

    "critical_marker_status":
        branch_c_marker_status,

    "all_critical_markers_preserved":
        all_critical_markers_preserved,

    "rotation_note_checks":
        branch_c_rotation_note_checks,

    "rotation_evidence_preserved":
        rotation_evidence_preserved,

    "numeric_token_preservation":
        numeric_preservation,

    "numeric_tokens_preserved":
        numeric_tokens_preserved,

    "complete_8_page_representation_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "page_removal_applied":
        False,

    "page_cropping_applied":
        False,

    "ocr_rerun_applied":
        False,

    "ocr_conversion_inherited_from_branch_B":
        True,

    "unicode_nfkc_normalisation_applied":
        True,

    "unicode_space_standardisation_applied":
        True,

    "apostrophe_standardisation_applied":
        True,

    "dash_and_minus_standardisation_applied":
        True,

    "soft_hyphen_removal_applied":
        True,

    "line_endings_standardised":
        True,

    "horizontal_whitespace_normalisation_applied":
        True,

    "paragraph_line_merging_applied":
        False,

    "line_break_hyphenation_repair_applied":
        False,

    "ocr_spelling_correction_applied":
        False,

    "semantic_harmonisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_integrity_passed
}


NORMALISATION_CHECK_PATH.write_text(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    )
)


if not normalisation_integrity_passed:
    raise ValueError(
        "D9 Branch C normalisation-integrity checks failed. "
        "Inspect parent equivalence, page/rotation evidence, "
        "critical markers and quantitative-token diagnostics."
    )


{
  "document_id": "D9",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "parent_page_markers": [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8"
  ],
  "branch_C_page_markers": [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8"
  ],
  "page_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "critical_marker_status": {
    "publication_title": true,
    "reference_year": true,
    "publication_year": true,
    "index_heading": true,
    "tabela_i": true,
    "tabela_ii": true,
    "portugal": true,
    "1911": true,
    "1920": true
  },
  "all_critical_markers_preserved": true,
  "rotation_note_checks": {
    "7": true,
    "8": true
  },
  "rotation_evidence_preserved": true,
  "numeric_token_preservation": {
    "four_digit_years": {
      "count_before": 24,
      "count_after": 24,
      "missing_token_count": 0,
      "added_token_count": 0,
      "passed": true
    },
    "com

In [11]:
# ============================================================
# 7. Save Branch C normalised representation
# ============================================================

REPRESENTATION_PATH.write_text(
    NORMALISED_MARKDOWN,
    encoding="utf-8"
)

REPRESENTATION_SHA256 = sha256_file(
    REPRESENTATION_PATH
)

print(
    "Saved:",
    REPRESENTATION_PATH.name
)

print(
    "Representation SHA-256:",
    REPRESENTATION_SHA256
)


Saved: D9_branch_C_normalised_markdown.md
Representation SHA-256: 2e8b6810c818f27b7d598083749b236eccea4271fc64d0d6138d101c89688c7c


In [12]:
# ============================================================
# 8. Create controlled Branch C extraction prompt
# ============================================================
#
# Substantive task and schema are frozen from final Branch B.
# Only the input-representation wording and branch wrapper identify C.
#
# Expected counts are deliberately NOT disclosed to the model.
# ============================================================

BRANCH_C_PROMPT = 'You are an information extraction assistant.\n\nExtract the predefined bibliographic, index, statistical and structural\nrecords represented within the defined scope of the attached deterministically\nnormalised OCR-based structural Markdown representation of the original\nhistorical scanned PDF publication:\n\n“Estatística do Movimento Fisiológico da População de Portugal —\nAno de 1925”.\n\nTreat the attached deterministically normalised structural Markdown\nrepresentation as the only source of information.\n\nThe supplied representation corresponds to the complete eight physical\npages of the scanned PDF subset. Do not infer information from omitted\nprinted pages of the larger historical publication.\n\nUse exactly these record fields:\n\n- Category\n- Topic\n- Description\n- Value\n- Unit\n- Reporting Period\n- Source Location\n\nUse exactly one of these Category values:\n\n- Publication metadata\n- Index entry\n- Statistical value\n- Document structure\n\n\n1. Publication metadata\n\nUsing the cover and imprint represented from physical PDF page 1,\nextract one record for each of these predefined topics:\n\n- Title\n- Reference year\n- Publication year\n- Publisher\n- Institution\n\nPreserve the represented Portuguese title and institution wording.\n\nKeep the statistical reference year distinct from the printed\npublication year.\n\nDo not infer bibliographic information that is not explicitly\nrepresented in the supplied source representation.\n\n\n2. Selected index entries\n\nUsing the index content represented from the supplied PDF, extract one\nrecord for each of these selected table identifiers:\n\n- Tabela I\n- Tabela II\n- Tabela III\n- Tabela XIV\n- Tabela LVIII\n- Tabela LIX\n\nFor each selected index entry:\n\n- preserve the Portuguese table identifier;\n- preserve the Portuguese table description as represented by OCR;\n- preserve the represented printed page number or page range in the\n  Source Location where recoverable;\n- preserve any explicitly associated period;\n- do not create a separate record from the accompanying French\n  translation.\n\nThe first four selected entries originate from physical PDF page 3.\nThe final two selected entries originate from physical PDF page 5.\n\n\n3. Selected statistical values from Tabela I\n\nFrom the Portugal row of Tabela I represented from physical PDF page 6,\nextract one record for each of these predefined indicators:\n\n- Portugal area\n- Portugal population 1911\n- Portugal population 1920\n- Portugal density 1920\n- Portugal average annual population growth\n\nRead each value directly from the represented Portugal row and its\ncorresponding OCR-recovered column context.\n\nPreserve the represented numeric scale.\n\nDo not calculate, derive, estimate, interpolate, rescale, correct or\nconvert any value.\n\nDo not extract district-level, city-level, sex-specific or other\nTabela I observations outside this predefined scope.\n\n\n4. Document structure\n\nCreate one source-grounded structural record for each of these\npredefined topics:\n\n- Rotated table\n- Bilingual headings\n- Historical typography\n\nFor “Rotated table”, use only structural evidence explicitly retained\nin the representation concerning the orientation and multi-page\npresentation of Tabela II.\n\nFor “Bilingual headings”, describe the represented relationship between\nthe Portuguese headings and their accompanying translated headings.\n\nFor “Historical typography”, describe only characteristics that remain\nexplicitly supported by the OCR-converted representation. Do not invent\nvisual details that are not preserved by the conversion.\n\nDo not infer any quantitative values from Tabela II.\n\n\nField rules:\n\nCategory:\n- Use exactly one of the four category labels defined above.\n\nTopic:\n- For each record, use the corresponding predefined topic label from\n  the scope above.\n\nDescription:\n- Provide a concise source-grounded description of the represented\n  record.\n- Preserve Portuguese source wording where the record is a publication\n  title or table description.\n- Do not add external interpretation.\n\nValue:\n- Use a JSON number when the source representation explicitly provides\n  a numeric value.\n- Use a JSON string when the represented value is textual.\n- Use null only when no separate Value is represented.\n- Preserve the represented printed scale.\n- Do not calculate, infer, derive, convert, repair or modernise values.\n\nUnit:\n- Preserve the explicitly associated measurement unit where recoverable.\n- Use null when no explicit unit applies.\n- Do not invent units.\n\nReporting Period:\n- Preserve an explicitly associated reference year or period.\n- Keep the statistical reference year and publication year distinct.\n- Preserve the census periods associated with the selected statistical\n  observations where explicitly represented.\n- Use null when no explicit reporting period applies.\n\nSource Location:\n- Use physical PDF page references exposed by the Markdown page\n  boundaries.\n- Where recoverable, also preserve the represented printed page or page\n  range for an index entry.\n- Use concise locations such as:\n  “PDF page 1 — Cover”\n  “PDF page 1 — Imprint”\n  “PDF page 3 — Índice; printed page ...”\n  “PDF page 5 — Índice; printed page ...”\n  “PDF page 6 — Tabela I, Portugal row”\n  “PDF pages 7–8 — Tabela II”\n\n\nAdditional extraction rules:\n\n- Use only information explicitly represented in the attached\n  structural Markdown.\n- Preserve Portuguese titles, table identifiers, historical spelling\n  and punctuation where recoverable.\n- Do not silently correct OCR wording or numerical values.\n- Do not treat French translations as separate duplicate records.\n- Do not use the original PDF, external OCR, external knowledge, or\n  information outside the attached representation.\n- Do not calculate, infer, derive, estimate or reconstruct missing\n  information.\n- Do not infer age-by-sex or other quantitative observations from\n  Tabela II.\n- Do not extract observations outside the predefined scope.\n- Ignore Markdown syntax and conversion labels except as structural\n  cues.\n- Verify that every item within the defined scope has been processed.\n- Return only valid JSON.\n- Do not include Markdown fences, explanations or commentary.\n- Keep the exact field names and field order defined below.\n\nExpected JSON structure:\n\n{\n  "document_id": "D9",\n  "branch": "C",\n  "records": [\n    {\n      "Category": null,\n      "Topic": null,\n      "Description": null,\n      "Value": null,\n      "Unit": null,\n      "Reporting Period": null,\n      "Source Location": null\n    }\n  ]\n}\n\nReturn only the JSON object.'

PROMPT_PATH.write_text(
    BRANCH_C_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)

print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)


Prompt saved: D9_branch_C_prompt.txt
Prompt SHA-256: 125f73c544f6bcc7171d60d411ed096afe87355d490a6b64ef9d17d22aba05a2


In [13]:
# ============================================================
# 9. Create representation and pre-extraction metadata
# ============================================================

REPRESENTATION_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "parent_B_representation_file":
        BRANCH_B_REPRESENTATION_PATH.name,

    "parent_B_representation_sha256":
        UPLOADED_BRANCH_B_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "parent_B_equivalence_method":
        "Frozen Branch B artefact SHA-256 verification; OCR not rerun",

    "representation_type":
        (
            "Complete frozen Branch B OCR structural Markdown "
            "with deterministic normalisation"
        ),

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "complete_8_page_representation_retained":
        True,

    "scope_enforced_by_prompt_not_representation_filtering":
        True,

    "ocr_conversion_inherited_from_branch_B":
        True,

    "ocr_rerun_in_branch_C":
        False,

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "normalisation_operations": [
        "Unicode NFKC normalisation",
        "Unicode-space standardisation",
        "apostrophe standardisation",
        "dash/minus-glyph standardisation",
        "soft-hyphen removal",
        "line-ending standardisation",
        "horizontal whitespace normalisation",
        "excessive blank-line standardisation"
    ],

    "semantic_rewriting_applied":
        False,

    "ocr_spelling_correction_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ]
}


REPRESENTATION_METADATA_PATH.write_text(
    json.dumps(
        REPRESENTATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_METADATA_PRE = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised OCR structural Markdown",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "direct_document_ingestion":
        False,

    "ocr_conversion_applied":
        True,

    "ocr_conversion_inherited_from_branch_B":
        True,

    "ocr_rerun_in_branch_C":
        False,

    "structural_conversion_applied":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "complete_source_document_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "reference_values_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "manual_response_repair_permitted":
        False,

    "expected_output_format":
        "JSON object",

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "execution_environment":
        "Independent ChatGPT conversation",

    "model":
        "GPT-5.5",

    "created_at":
        datetime.now().isoformat(),

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "validation_status":
        "Pending independent Branch C extraction and Stage 4 Validation C"
}


EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D9",
  "document_name": "Estatística do Movimento Fisiológico da População de Portugal — Ano de 1925",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D9 - EMovimentoFisiológico1925 (1).pdf",
  "source_sha256": "a8fd4297949fa5b75b6e801ecd95aa9f7c15f9b08c789bc530f2266f41106cc1",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised OCR structural Markdown",
  "representation_file": "D9_branch_C_normalised_markdown.md",
  "representation_sha256": "2e8b6810c818f27b7d598083749b236eccea4271fc64d0d6138d101c89688c7c",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "direct_document_ingestion": false,
  "ocr_conversion_applied": true,
  "ocr_conversion_inherited_from_branch_B": true,
  "ocr_rerun_in_branch_C": false,
  "structural_conversion_applied": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true

In [14]:
# ============================================================
# 10. Final pre-extraction control check
# ============================================================

PRECHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_identity_verified":
        SOURCE_HASH_MATCH,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "complete_8_page_representation_retained":
        True,

    "page_sequence_preserved":
        page_sequence_preserved,

    "rotation_evidence_preserved":
        rotation_evidence_preserved,

    "all_critical_markers_preserved":
        all_critical_markers_preserved,

    "representation_exists":
        REPRESENTATION_PATH.exists(),

    "prompt_exists":
        PROMPT_PATH.exists(),

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "ready_for_independent_llm_execution":
        bool(
            SOURCE_HASH_MATCH
            and PARENT_EQUIVALENCE_PASSED
            and normalisation_check[
                "normalisation_integrity_passed"
            ]
            and REPRESENTATION_PATH.exists()
            and PROMPT_PATH.exists()
        )
}


PRECHECK_PATH.write_text(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    )
)


if not PRECHECK[
    "ready_for_independent_llm_execution"
]:
    raise ValueError(
        "D9 Branch C is not ready for independent LLM execution."
    )


{
  "document_id": "D9",
  "branch": "C",
  "source_identity_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "complete_8_page_representation_retained": true,
  "page_sequence_preserved": true,
  "rotation_evidence_preserved": true,
  "all_critical_markers_preserved": true,
  "representation_exists": true,
  "prompt_exists": true,
  "expected_record_count_disclosed_to_model": false,
  "expected_category_counts_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [15]:
# ============================================================
# 11. Download pre-extraction Branch C artefacts
# ============================================================

for path in [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH
]:
    files.download(
        path
    )


print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D9_branch_C_normalised_markdown.md.\n"
    "3. Submit D9_branch_C_prompt.txt exactly once.\n"
    "4. Do not upload the original PDF, Branch B artefacts, Stage 1 "
    "reference values, or previous extraction outputs.\n"
    "5. Do not run external OCR or consult the original PDF during extraction.\n"
    "6. Do not manually repair, correct, or regenerate the response.\n"
    "7. Save the complete response exactly as returned in a plain-text file."
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Independent extraction instructions:
1. Open a new independent ChatGPT conversation.
2. Upload ONLY D9_branch_C_normalised_markdown.md.
3. Submit D9_branch_C_prompt.txt exactly once.
4. Do not upload the original PDF, Branch B artefacts, Stage 1 reference values, or previous extraction outputs.
5. Do not run external OCR or consult the original PDF during extraction.
6. Do not manually repair, correct, or regenerate the response.
7. Save the complete response exactly as returned in a plain-text file.


In [16]:
# ============================================================
# 12. Upload and preserve the complete raw Branch C response
# ============================================================

uploaded_response = files.upload()

if len(uploaded_response) != 1:
    raise ValueError(
        "Upload exactly one complete raw Branch C response file."
    )


RAW_RESPONSE_SOURCE = Path(
    next(
        iter(
            uploaded_response
        )
    )
)


RAW_RESPONSE_TEXT = (
    RAW_RESPONSE_SOURCE.read_text(
        encoding="utf-8"
    )
)


RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)


print(
    "Raw response preserved unchanged."
)

print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)


Saving D9_branch_C_raw_response.txt to D9_branch_C_raw_response.txt
Raw response preserved unchanged.
Raw response SHA-256: 3bcd0fd36b238d644e15f9d20a50f59b903069e2fabdb1b7495eb242ba01e04c


In [17]:
# ============================================================
# 13. Parse raw response WITHOUT repair
# ============================================================

valid_json = True
json_parsing_error = None
parsed_response = None


try:
    parsed_response = json.loads(
        RAW_RESPONSE_TEXT
    )

except json.JSONDecodeError as exc:
    valid_json = False
    json_parsing_error = str(exc)


top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)

document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)

document_id_correct = (
    document_id_present
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)

branch_correct = (
    branch_present
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)

records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)

records_is_list = (
    records_present
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)

records_evaluable = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
)

extracted_records = (
    parsed_response["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(
        extracted_records
    )
    if records_evaluable
    else None
)


print(
    "Valid JSON:",
    valid_json
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed records:",
    observed_record_count
)

if json_parsing_error:
    print(
        "JSON parsing error:",
        json_parsing_error
    )


Valid JSON: False
Records evaluable: False
Observed records: None
JSON parsing error: Expecting ',' delimiter: line 170 column 118 (char 5359)


In [18]:
# ============================================================
# 14. Validate record schema and field types
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []


if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):
            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Record is not a JSON object"
            })

            continue


        observed_fields = list(
            record.keys()
        )

        if observed_fields != EXPECTED_FIELDS:
            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Field names or field order differ",

                "expected_fields":
                    EXPECTED_FIELDS,

                "observed_fields":
                    observed_fields
            })


        for field in MANDATORY_STRING_FIELDS:

            value = record.get(
                field
            )

            if (
                value is None
                or (
                    isinstance(
                        value,
                        str
                    )
                    and not value.strip()
                )
            ):
                missing_mandatory_values.append({
                    "record_index":
                        record_index,

                    "field":
                        field
                })

            elif not isinstance(
                value,
                str
            ):
                field_type_issues.append({
                    "record_index":
                        record_index,

                    "field":
                        field,

                    "observed_type":
                        type(
                            value
                        ).__name__
                })


        for field in NULLABLE_STRING_FIELDS:

            value = record.get(
                field
            )

            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):
                field_type_issues.append({
                    "record_index":
                        record_index,

                    "field":
                        field,

                    "observed_type":
                        type(
                            value
                        ).__name__
                })


        value = record.get(
            "Value"
        )

        if (
            isinstance(
                value,
                bool
            )
            or not isinstance(
                value,
                ALLOWED_VALUE_TYPES
            )
        ):
            field_type_issues.append({
                "record_index":
                    record_index,

                "field":
                    "Value",

                "observed_type":
                    type(
                        value
                    ).__name__
            })


record_schema_valid = (
    len(
        record_structure_issues
    )
    == 0
    if records_evaluable
    else None
)

field_types_valid = (
    len(
        field_type_issues
    )
    == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(
        missing_mandatory_values
    )
    == 0
    if records_evaluable
    else None
)


print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Mandatory fields complete:",
    mandatory_fields_complete
)

print(
    "Structure issues:",
    len(
        record_structure_issues
    )
)

print(
    "Type issues:",
    len(
        field_type_issues
    )
)


Record schema valid: None
Field types valid: None
Mandatory fields complete: None
Structure issues: 0
Type issues: 0


In [19]:
# ============================================================
# 15. Content/scope diagnostics kept separate from schema validity
# ============================================================

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get(
                "Category"
            )
            for record
            in extracted_records
            if isinstance(
                record,
                dict
            )
        )
    )


    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


    observed_topics_by_category = {
        category: {
            record.get("Topic")
            for record
            in extracted_records
            if (
                isinstance(record, dict)
                and record.get("Category") == category
            )
        }
        for category
        in EXPECTED_TOPICS
    }


    topic_scope_checks = {
        category:
            (
                observed_topics_by_category.get(
                    category,
                    set()
                )
                == expected_topics
            )
        for category, expected_topics
        in EXPECTED_TOPICS.items()
    }


    all_expected_topics_present = all(
        topic_scope_checks.values()
    )


    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(field),
                ensure_ascii=False,
                sort_keys=True
            )
            for field
            in EXPECTED_FIELDS
        )
        for record
        in extracted_records
        if isinstance(
            record,
            dict
        )
    )


    duplicate_complete_records = [
        {
            "record":
                list(
                    key
                ),

            "occurrence_count":
                count
        }
        for key, count
        in duplicate_counter.items()
        if count > 1
    ]


    duplicate_complete_record_count = len(
        duplicate_complete_records
    )


    def find_topic_record(topic):
        matches = [
            record
            for record
            in extracted_records
            if (
                isinstance(record, dict)
                and record.get("Topic") == topic
            )
        ]

        return (
            matches[0]
            if len(matches) == 1
            else None
        )


    reference_year_record = find_topic_record(
        "Reference year"
    )

    publication_year_record = find_topic_record(
        "Publication year"
    )


    year_distinction_preserved = bool(
        reference_year_record is not None
        and publication_year_record is not None
        and reference_year_record.get(
            "Value"
        ) == 1925
        and publication_year_record.get(
            "Value"
        ) == 1929
    )


    tabela_ii_quantitative_records = [
        record
        for record
        in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Category") == "Statistical value"
            and isinstance(
                record.get("Source Location"),
                str
            )
            and (
                "page 7"
                in record.get(
                    "Source Location"
                ).casefold()
                or "page 8"
                in record.get(
                    "Source Location"
                ).casefold()
            )
        )
    ]


    no_tabela_ii_quantitative_extraction = (
        len(
            tabela_ii_quantitative_records
        )
        == 0
    )


else:

    record_count_valid = None
    observed_category_counts = None
    categories_valid = None
    category_counts_valid = None
    observed_topics_by_category = None
    topic_scope_checks = None
    all_expected_topics_present = None
    duplicate_complete_records = None
    duplicate_complete_record_count = None
    year_distinction_preserved = None
    tabela_ii_quantitative_records = None
    no_tabela_ii_quantitative_extraction = None


scope_complete = bool(
    record_count_valid
    and category_counts_valid
    and all_expected_topics_present
) if records_evaluable else False


CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches_reference":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "topic_scope_checks":
        topic_scope_checks,

    "all_expected_topics_present":
        all_expected_topics_present,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        (
            len(
                missing_mandatory_values
            )
            if records_evaluable
            else None
        ),

    "duplicate_complete_record_count":
        duplicate_complete_record_count,

    "duplicate_complete_records":
        duplicate_complete_records,

    "duplicate_check_is_diagnostic_only":
        True,

    "year_distinction_preserved":
        year_distinction_preserved,

    "tabela_II_quantitative_record_count":
        (
            len(
                tabela_ii_quantitative_records
            )
            if records_evaluable
            else None
        ),

    "no_tabela_II_quantitative_extraction":
        no_tabela_ii_quantitative_extraction
}


print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


{
  "expected_record_count": 19,
  "observed_record_count": null,
  "record_count_matches_reference": null,
  "expected_category_counts": {
    "Publication metadata": 5,
    "Index entry": 6,
    "Statistical value": 5,
    "Document structure": 3
  },
  "observed_category_counts": null,
  "categories_valid": null,
  "category_counts_match_reference": null,
  "topic_scope_checks": null,
  "all_expected_topics_present": null,
  "mandatory_fields_complete": null,
  "missing_mandatory_value_count": null,
  "duplicate_complete_record_count": null,
  "duplicate_complete_records": null,
  "duplicate_check_is_diagnostic_only": true,
  "year_distinction_preserved": null,
  "tabela_II_quantitative_record_count": null,
  "no_tabela_II_quantitative_extraction": null
}


In [20]:
# ============================================================
# 16. Determine technical/schema validity
# ============================================================
#
# Count/category/topic agreement are CONTENT/SCOPE diagnostics and
# are deliberately not conditions for technical schema validity.
# ============================================================

structure_valid = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
    and record_schema_valid is True
    and field_types_valid is True
)


STRUCTURE_CHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        "Complete deterministically normalised OCR structural Markdown",

    "valid_json":
        bool(
            valid_json
        ),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(
            top_level_object_valid
        ),

    "document_id_correct":
        bool(
            document_id_correct
        ),

    "branch_correct":
        bool(
            branch_correct
        ),

    "records_is_list":
        bool(
            records_is_list
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structure_valid":
        bool(
            structure_valid
        ),

    "scope_complete":
        bool(
            scope_complete
        )
}


STRUCTURE_CHECK_PATH.write_text(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D9",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised OCR structural Markdown",
  "valid_json": false,
  "json_parsing_error": "Expecting ',' delimiter: line 170 column 118 (char 5359)",
  "top_level_object_valid": false,
  "document_id_correct": false,
  "branch_correct": false,
  "records_is_list": false,
  "records_evaluable": false,
  "record_schema_valid": null,
  "record_structure_issues": null,
  "field_types_valid": null,
  "field_type_issues": null,
  "content_diagnostics": {
    "expected_record_count": 19,
    "observed_record_count": null,
    "record_count_matches_reference": null,
    "expected_category_counts": {
      "Publication metadata": 5,
      "Index entry": 6,
      "Statistical value": 5,
      "Document structure": 3
    },
    "observed_category_counts": null,
    "categories_valid": null,
    "category_counts_match_reference": null,
    "topic_scope_checks": 

In [21]:
# ============================================================
# 17. Preserve parsed extraction only when records are evaluable
# ============================================================

parsed_extraction_created = False
parsed_extraction_sha256 = None


if records_evaluable:

    canonical_extraction = {
        "document_id":
            DOCUMENT_ID,

        "branch":
            BRANCH,

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            canonical_extraction,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )

    parsed_extraction_created = True


    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH.name
    )

else:

    print(
        "No parsed extraction created because the preserved raw response "
        "does not contain an evaluable records structure."
    )


No parsed extraction created because the preserved raw response does not contain an evaluable records structure.


In [22]:
# ============================================================
# 18. Create final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "structure_check_file":
        STRUCTURE_CHECK_PATH.name,

    "structure_valid":
        bool(
            structure_valid
        ),

    "notes": (
        "Branch C applies deterministic non-semantic normalisation to the "
        "exact frozen Branch B OCR structural representation. OCR is not "
        "rerun in Branch C. The complete eight-page representation is "
        "retained, expected Stage 1 counts are not supplied to the model, "
        "and accuracy is evaluated separately in Validation C."
    )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised OCR structural Markdown",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "ocr_conversion_inherited_from_branch_B":
        True,

    "ocr_rerun_in_branch_C":
        False,

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "complete_source_document_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "raw_response_preserved":
        True,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "valid_json":
        bool(
            valid_json
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "record_schema_valid":
        record_schema_valid,

    "field_types_valid":
        field_types_valid,

    "structure_valid":
        bool(
            structure_valid
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "all_expected_topics_present":
        all_expected_topics_present,

    "scope_complete":
        bool(
            scope_complete
        ),

    "duplicate_complete_record_count":
        duplicate_complete_record_count,

    "year_distinction_preserved":
        year_distinction_preserved,

    "no_tabela_II_quantitative_extraction":
        no_tabela_ii_quantitative_extraction,

    "parsed_extraction_created":
        parsed_extraction_created,

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "accuracy_validation_completed":
        False,

    "validation_status":
        (
            "Pending Stage 4 Branch C validation against the fixed Stage 1 "
            "reference dataset using Branch A-frozen D9 comparison rules"
            if records_evaluable
            else
            "Not content-evaluable because the preserved Branch C response "
            "does not contain an evaluable records structure"
        )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D9",
  "document_name": "Estatística do Movimento Fisiológico da População de Portugal — Ano de 1925",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D9 - EMovimentoFisiológico1925 (1).pdf",
  "source_sha256": "a8fd4297949fa5b75b6e801ecd95aa9f7c15f9b08c789bc530f2266f41106cc1",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised OCR structural Markdown",
  "representation_file": "D9_branch_C_normalised_markdown.md",
  "representation_sha256": "2e8b6810c818f27b7d598083749b236eccea4271fc64d0d6138d101c89688c7c",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "ocr_conversion_inherited_from_branch_B": true,
  "ocr_rerun_in_branch_C": false,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_source_document_retained": true,
  "source_scope_filtering_applied": false,
  "reference_values_use

In [23]:
# ============================================================
# 19. Final artefact inventory and download
# ============================================================

artefacts = [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH,
    RAW_RESPONSE_PATH,
    STRUCTURE_CHECK_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    artefacts.append(
        PARSED_EXTRACTION_PATH
    )


print(
    "Final D9 Branch C artefacts:"
)

for path in artefacts:
    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )


for path in artefacts:
    if path.exists():
        files.download(
            path
        )


Final D9 Branch C artefacts:
- D9_branch_C_parent_B_equivalence_check.json | exists: True
- D9_branch_C_normalisation_check.json | exists: True
- D9_branch_C_normalised_markdown.md | exists: True
- D9_branch_C_representation_metadata.json | exists: True
- D9_branch_C_prompt.txt | exists: True
- D9_branch_C_experiment_metadata_pre.json | exists: True
- D9_branch_C_pre_extraction_check.json | exists: True
- D9_branch_C_raw_response.txt | exists: True
- D9_branch_C_structure_check.json | exists: True
- D9_branch_C_experiment_metadata.json | exists: True
- D9_branch_C_experiment_summary.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>